In [2]:
import pandas as pd
import numpy as np

# Exploratory Analysis

In [ ]:
## Load the data with the target columns
# !cut -f 1,8,15,18,21,22,26,67,68,71 /home/camilababo/Documents/DNAquaIMG/BOLD_Public.29-Nov-2024/BOLD_Public.29-Nov-2024.tsv | gzip -c - > bold-selected.tsv.gz

In [3]:
# Load the data
path = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/external_scripts/bold-selected.tsv"
bold = pd.read_csv(path, on_bad_lines="skip", sep="\t")

/tmp/ipykernel_10207/1837559620.py:2: DtypeWarning: Columns (1,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  bold = pd.read_csv(path, on_bad_lines="skip", sep="\t")


In [42]:
bold.head()

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
0,AANIC003-10,BOLD:AAB9307,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
1,AANIC002-10,BOLD:AAO2553,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCAGGTATAATTGGAACT...,658.0,COI-5P
2,AANIC058-10,BOLD:AAB9307,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATCTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
3,AANIC061-10,BOLD:ACE8261,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
4,AANIC062-10,BOLD:AAF6782,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P


In [4]:
bold.tail()

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
17862992,ZYGMO209-10,BOLD:AAJ5548,Arthropoda,Zygaenidae,Adscita,Adscita obscura,NaN,AACACTTTACTTTATTTTTGGTATTTGATCAGGAATAGTTGGAACA...,658.0,COI-5P
17862993,ZYGMO290-10,BOLD:AAO1273,Arthropoda,Zygaenidae,Rhagades,Rhagades predotae,NaN,----------------------------------------------...,608.0,COI-5P
17862994,ZYGMO294-10,BOLD:AAO0371,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris minna,NaN,AACACTTTATTTTATTTTTGGAATTTGATCAGGAATAATTGGTACA...,658.0,COI-5P
17862995,ZYGMO295-10,BOLD:AAN9389,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris rjabovi,NaN,AACACTTTATTTTATCTTTGGAATCTGATCAGGAATAATTGGAACA...,658.0,COI-5P
17862996,ZYGMO296-10,BOLD:AAN9389,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris rjabovi,NaN,AACACTTTATTTTATCTTTGGAATCTGATCAGGAATAATTGGAACA...,658.0,COI-5P


In [17]:
# Check the total number of entries
bold.shape

(17862997, 10)

#### Check entries on 'Identification Method'

In [15]:
non_null_id = bold[bold['identification_method'] == 'Wikipidia']
non_null_id

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
654690,AZSDS020-18,NaN,Arthropoda,Cerambycidae,NaN,NaN,Wikipidia,NaN,NaN,NaN


In [18]:
# Check the number of entries with COI-5P
count = bold['marker_code'].str.contains('COI-5P', na=False).sum()
int(count)

14790363

In [19]:
# Check the average length of entrie's nucleotide sequence
average_length = bold['nuc'].astype(str).apply(len).mean()
int(average_length)

610

In [13]:
# Check values with unique entries for 'identification method'
unique_idm = bold['identification_method'].unique()
np.savetxt("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/external_scripts/idm.txt",
           unique_idm, fmt='%s')

In [ ]:
# Check values with unique entries for 'marker code'
unique_mc = bold['marker_code'].unique()
np.savetxt("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/external_scripts/mc.txt",
           unique_mc, fmt='%s')

## Pre-processing

#### Remove all entries that do not have at least family-level info

In [44]:
species_lev = bold['family'].isna()

In [46]:
bold  = bold[~species_lev]

#### Create column indicating highest taxon level between 'family' and 'species'

In [51]:
bold['highest_tax_level'] = bold[['species', 'genus', 'family']].bfill(axis=1).iloc[:, 0]

/tmp/ipykernel_10207/2600937051.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bold['highest_tax_level'] = bold[['species', 'genus', 'family']].bfill(axis=1).iloc[:, 0]


In [52]:
bold

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code,highest_tax_level
0,AANIC003-10,BOLD:AAB9307,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P,Arhodia lasiocamparia
1,AANIC002-10,BOLD:AAO2553,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCAGGTATAATTGGAACT...,658.0,COI-5P,Arhodia lasiocamparia
2,AANIC058-10,BOLD:AAB9307,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATCTGAGCTGGTATAATTGGAACT...,658.0,COI-5P,Arhodia lasiocamparia
3,AANIC061-10,BOLD:ACE8261,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P,Arhodia lasiocamparia
4,AANIC062-10,BOLD:AAF6782,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P,Arhodia lasiocamparia
...,...,...,...,...,...,...,...,...,...,...,...
17862992,ZYGMO209-10,BOLD:AAJ5548,Arthropoda,Zygaenidae,Adscita,Adscita obscura,NaN,AACACTTTACTTTATTTTTGGTATTTGATCAGGAATAGTTGGAACA...,658.0,COI-5P,Adscita obscura
17862993,ZYGMO290-10,BOLD:AAO1273,Arthropoda,Zygaenidae,Rhagades,Rhagades predotae,NaN,----------------------------------------------...,608.0,COI-5P,Rhagades predotae
17862994,ZYGMO294-10,BOLD:AAO0371,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris minna,NaN,AACACTTTATTTTATTTTTGGAATTTGATCAGGAATAATTGGTACA...,658.0,COI-5P,Zygaenoprocris minna
17862995,ZYGMO295-10,BOLD:AAN9389,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris rjabovi,NaN,AACACTTTATTTTATCTTTGGAATCTGATCAGGAATAATTGGAACA...,658.0,COI-5P,Zygaenoprocris rjabovi


In [59]:
bold[bold['species'].isna()]

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code,highest_tax_level
33,AAPY056-10,BOLD:AAM5892,Arthropoda,Araneidae,NaN,NaN,NaN,AACATTATATTTAGTATTTGGGGCCTGATCAGCAATGGTTGGGACT...,658.0,COI-5P,Araneidae
34,AAPY058-10,BOLD:AAZ0791,Arthropoda,Araneidae,NaN,NaN,NaN,AACATTATATTTAGTATTTGGAGCTTGAGCTGCTATAGTAGGAACT...,658.0,COI-5P,Araneidae
35,AAPY059-10,BOLD:AAZ0381,Arthropoda,Araneidae,Araneus,NaN,NaN,TACGTTATATTTGATTTTTGGGGCTTGATCGGCCATAATTGGTACA...,658.0,COI-5P,Araneus
36,AAPY060-10,BOLD:AAM5892,Arthropoda,Araneidae,Araneus,NaN,NaN,AACATTATATTTAGTATTTGGGGCCTGATCAGCAATGGTTGGGACT...,658.0,COI-5P,Araneus
37,AAPY061-10,BOLD:AAM5892,Arthropoda,Araneidae,NaN,NaN,NaN,AACATTATATTTAGTATTTGGGGCCTGATCAGCAATGGTTGGGACT...,658.0,COI-5P,Araneidae
...,...,...,...,...,...,...,...,...,...,...,...
17862918,ZSIVP269-24,BOLD:AFS2012,Chordata,Nemacheilidae,Schistura,NaN,NaN,CTTGGTGATGACCAAATTTACAATGTTATTGTTACCGCACACGCTT...,582.0,COI-5P,Schistura
17862929,ZSIVP267-24,BOLD:AFS2012,Chordata,Nemacheilidae,Schistura,NaN,NaN,CTTGGTGATGACCAAATTTACAATGTTATTGTTACCGCACACGCTT...,582.0,COI-5P,Schistura
17862930,ZSIVP268-24,BOLD:AFS2012,Chordata,Nemacheilidae,Schistura,NaN,NaN,CTTGGTGATGACCAAATTTACAATGTTATTGTTACCGCACACGCTT...,582.0,COI-5P,Schistura
17862953,ZSIVP301-24,BOLD:AFT5369,Chordata,Cyprinidae,Garra,NaN,NaN,ATGACCAAATTTATAATGTACGTTACTGCCCATGCCTTTGTAATAA...,573.0,COI-5P,Garra
